In [ ]:
from pathlib import Path
import sys
import sklearn, numpy, pandas

REPO_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "ai-models" / "data" / "dataset.zip").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("KhÃƒÂ´ng tÃƒÂ¬m thÃ¡ÂºÂ¥y ai-models/data/dataset.zip trong repo hiÃ¡Â»â€¡n tÃ¡ÂºÂ¡i")

sys.path.insert(0, str(REPO_ROOT / "ai-models" / "src"))
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("Repo:", REPO_ROOT)

In [ ]:
import ast, pandas as pd
from preprocess import ROOT

results = pd.read_csv(ROOT / "docs" / "model_comparison.csv")

MODEL_NAME = "Logistic Regression (baseline)"   # Ã„â€˜Ã¡Â»â€¢i thÃƒÂ nh "SVM (RBF)" nÃ¡ÂºÂ¿u bÃ¡ÂºÂ¡n chÃ¡Â»Ân SVM

row = results[results["model"] == MODEL_NAME].iloc[0]
best_params = ast.literal_eval(row["best_params"])
print("Model chÃ¡Â»Ân:", MODEL_NAME)
print("SiÃƒÂªu tham sÃ¡Â»â€˜:", best_params)
print("Test F1 lÃƒÂºc Ã„â€˜ÃƒÂ¡nh giÃƒÂ¡:", row["test_f1"], "| Test Recall:", row["test_recall"])

In [ ]:
from preprocess import load_data, make_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

X, y = load_data()   # dÃƒÂ¹ng cÃ¡ÂºÂ£ 569 mÃ¡ÂºÂ«u, khÃƒÂ´ng chia train/test nÃ¡Â»Â¯a

if MODEL_NAME.startswith("Logistic"):
    clf = LogisticRegression(max_iter=5000, class_weight="balanced",
                             random_state=42, **best_params)
elif MODEL_NAME.startswith("SVM"):
    clf = SVC(kernel="rbf", probability=True, class_weight="balanced",
             random_state=42, **best_params)
else:
    raise ValueError("ThÃƒÂªm nhÃƒÂ¡nh cho model nÃƒÂ y nÃ¡ÂºÂ¿u bÃ¡ÂºÂ¡n chÃ¡Â»Ân KNN/Decision Tree/Random Forest")

final_model = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
final_model.fit(X, y)
print("Ã„ÂÃƒÂ£ huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n xong trÃƒÂªn", X.shape[0], "mÃ¡ÂºÂ«u")

In [ ]:
sample = X.iloc[[0]]
pred = final_model.predict(sample)[0]
proba = final_model.predict_proba(sample)[0, 1]
print("DÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n mÃ¡ÂºÂ«u Ã„â€˜Ã¡ÂºÂ§u tiÃƒÂªn:", "malignant" if pred == 1 else "benign", "| xÃƒÂ¡c suÃ¡ÂºÂ¥t ÃƒÂ¡c tÃƒÂ­nh:", round(proba, 4))

In [ ]:
import joblib
from preprocess import MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "model.joblib"
joblib.dump(final_model, model_path, compress=3)
print("Ã„ÂÃƒÂ£ lÃ†Â°u:", model_path, "-", round(model_path.stat().st_size / 1024, 1), "KB")

In [ ]:
import json, datetime, sklearn, numpy, pandas

metadata = {
    "model_name": MODEL_NAME,
    "model_version": "1.0.0",
    "trained_on": datetime.date.today().isoformat(),
    "hyperparameters": best_params,
    "metrics_at_selection": {          # Ã„â€˜o trÃƒÂªn tÃ¡ÂºÂ­p test 20%, lÃƒÂºc so sÃƒÂ¡nh model
        "test_recall": float(row["test_recall"]),
        "test_precision": float(row["test_precision"]),
        "test_f1": float(row["test_f1"]),
        "test_roc_auc": float(row["test_roc_auc"]),
    },
    "trained_on_full_dataset": True,   # bÃ¡ÂºÂ£n lÃ†Â°u nÃƒÂ y hÃ¡Â»Âc trÃƒÂªn toÃƒÂ n bÃ¡Â»â„¢ 569 mÃ¡ÂºÂ«u
    "library_versions": {
        "scikit-learn": sklearn.__version__,
        "numpy": numpy.__version__,
        "pandas": pandas.__version__,
    },
    "positive_class": "malignant",
    "labels": {"0": "benign", "1": "malignant"},
}

with open(MODELS_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

In [ ]:
print("Model Ã„â€˜ÃƒÂ£ Ã„â€˜ÃƒÂ³ng gÃƒÂ³i tÃ¡ÂºÂ¡i:", MODELS_DIR / "model.joblib")
print("Metadata Ã„â€˜ÃƒÂ£ lÃ†Â°u tÃ¡ÂºÂ¡i:", MODELS_DIR / "metadata.json")

## Ã„ÂÃƒÂ³ng gÃƒÂ³i model

**Model Ã„â€˜Ã†Â°Ã¡Â»Â£c chÃ¡Â»Ân:** Logistic Regression, C = 0.1 (xem lÃƒÂ½ do chÃ¡Â»Ân Ã¡Â»Å¸ notebook 04).

**VÃ¡Â»â€¹ trÃƒÂ­ trong repo:** `ai-models/models/model.joblib`. File chÃ¡Â»Â©a cÃ¡ÂºÂ£ pipeline tiÃ¡Â»Ân xÃ¡Â»Â­ lÃƒÂ½ (SimpleImputer + StandardScaler) vÃƒÂ  model, nÃƒÂªn khi dÃ¡Â»Â± Ã„â€˜oÃƒÂ¡n chÃ¡Â»â€° cÃ¡ÂºÂ§n gÃ¡Â»Âi `model.predict()` trÃƒÂªn dÃ¡Â»Â¯ liÃ¡Â»â€¡u thÃƒÂ´, khÃƒÂ´ng cÃ¡ÂºÂ§n xÃ¡Â»Â­ lÃƒÂ½ tay.

**CÃƒÂ¡ch export tÃ¡Â»Â« Colab:** chÃ¡ÂºÂ¡y `ai-models/colab/05_package.ipynb`, huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n lÃ¡ÂºÂ¡i model Ã„â€˜ÃƒÂ£ chÃ¡Â»Ân trÃƒÂªn toÃƒÂ n bÃ¡Â»â„¢ 569 mÃ¡ÂºÂ«u (dÃƒÂ¹ng Ã„â€˜ÃƒÂºng siÃƒÂªu tham sÃ¡Â»â€˜ Ã„â€˜ÃƒÂ£ tÃƒÂ¬m Ã„â€˜Ã†Â°Ã¡Â»Â£c Ã¡Â»Å¸ bÃ†Â°Ã¡Â»â€ºc tinh chÃ¡Â»â€°nh), rÃ¡Â»â€œi tÃ¡ÂºÂ£i `model.joblib` vÃƒÂ  `metadata.json` vÃ¡Â»Â mÃƒÂ¡y vÃƒÂ  commit lÃƒÂªn Git. AI Service Ã„â€˜Ã¡Â»Âc trÃ¡Â»Â±c tiÃ¡ÂºÂ¿p file nÃƒÂ y khi container khÃ¡Â»Å¸i Ã„â€˜Ã¡Â»â„¢ng.

**PhiÃƒÂªn bÃ¡ÂºÂ£n thÃ†Â° viÃ¡Â»â€¡n lÃƒÂºc huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n:** scikit-learn 1.6.1, numpy 2.1.3 pandas 2.2.3

. CÃƒÂ¡c phiÃƒÂªn bÃ¡ÂºÂ£n nÃƒÂ y Ã„â€˜Ã†Â°Ã¡Â»Â£c ghim trong `ai-models/requirements.txt` Ã„â€˜Ã¡Â»Æ’ mÃƒÂ´i trÃ†Â°Ã¡Â»Âng Docker khÃ¡Â»â€ºp vÃ¡Â»â€ºi lÃƒÂºc huÃ¡ÂºÂ¥n luyÃ¡Â»â€¡n.